In [68]:
print("Welcome to StudyPlanner AI!")
print("Our project setup is working.")

Welcome to StudyPlanner AI!
Our project setup is working.


In [69]:
import sqlite3

# Create or open our database file
conn = sqlite3.connect("studyplanner.db")
cursor = conn.cursor()

# Create students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    daily_hours REAL NOT NULL
)
""")

# Create study tasks table
cursor.execute("""
CREATE TABLE IF NOT EXISTS study_tasks (
    task_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    subject TEXT NOT NULL,
    topic TEXT NOT NULL,
    exam_date TEXT NOT NULL,
    duration_minutes INTEGER NOT NULL,
    priority TEXT NOT NULL,
    completed INTEGER DEFAULT 0,
    FOREIGN KEY (student_id)
        REFERENCES students(student_id)
)
""")

conn.commit()

print("Database created successfully!")
print("Tables: students, study_tasks")

Database created successfully!
Tables: students, study_tasks


In [70]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

print(cursor.fetchall())

[('planned_sessions',), ('sqlite_sequence',), ('students',), ('study_tasks',)]


In [71]:
# Add one demo student
cursor.execute("""
INSERT INTO students (name, daily_hours)
VALUES (?, ?)
""", ("Demo Student", 3))

student_id = cursor.lastrowid

# Add three study tasks
tasks = [
    (student_id, "Python", "Loops", "2026-10-15", 45, "High"),
    (student_id, "Mathematics", "Matrices", "2026-10-17", 60, "Medium"),
    (student_id, "DBMS", "SQL Basics", "2026-10-19", 45, "High")
]

cursor.executemany("""
INSERT INTO study_tasks
(student_id, subject, topic, exam_date,
 duration_minutes, priority)
VALUES (?, ?, ?, ?, ?, ?)
""", tasks)

conn.commit()

print("Sample student and study tasks added!")

Sample student and study tasks added!


In [72]:
cursor.execute("""
SELECT
    s.name,
    t.subject,
    t.topic,
    t.exam_date,
    t.duration_minutes,
    t.priority,
    t.completed
FROM students AS s
JOIN study_tasks AS t
ON s.student_id = t.student_id
WHERE s.student_id = ?
""", (student_id,))

for row in cursor.fetchall():
    print(row)

('Demo Student', 'Python', 'Loops', '2026-10-15', 45, 'High', 0)
('Demo Student', 'Mathematics', 'Matrices', '2026-10-17', 60, 'Medium', 0)
('Demo Student', 'DBMS', 'SQL Basics', '2026-10-19', 45, 'High', 0)


In [73]:
# Get the first task for our demo student
cursor.execute("""
SELECT task_id, subject, topic
FROM study_tasks
WHERE student_id = ?
ORDER BY task_id
LIMIT 1
""", (student_id,))

task = cursor.fetchone()

# Mark the task as completed
cursor.execute("""
UPDATE study_tasks
SET completed = 1
WHERE task_id = ?
""", (task[0],))

conn.commit()

print("Completed task:", task[1], "-", task[2])

Completed task: Python - Loops


In [74]:

from datetime import date, timedelta

def generate_study_plan(subjects, daily_hours):
    """
    subjects is a list of dictionaries containing:
    subject, topic, exam_date, difficulty
    """

    today = date.today()
    available_minutes = int(daily_hours * 60)

    # Prioritize difficult topics and approaching exams
    subjects = sorted(
        subjects,
        key=lambda s: (
            date.fromisoformat(s["exam_date"]) - today,
            -s["difficulty"]
        )
    )

    plan = []
    study_date = today
    daily_used = 0

    for item in subjects:
        exam_day = date.fromisoformat(item["exam_date"])

        if exam_day < today:
            continue

        # Start a new day when today's available time is used
        if daily_used + 45 > available_minutes:
            study_date += timedelta(days=1)
            daily_used = 0

        # Do not schedule study after the exam
        while study_date > exam_day:
            study_date = today
            daily_used = 0
            break

        if study_date > exam_day:
            continue

        plan.append({
            "date": study_date.isoformat(),
            "subject": item["subject"],
            "topic": item["topic"],
            "duration_minutes": 45,
            "priority": (
                "High" if item["difficulty"] >= 4
                else "Medium"
            )
        })

        daily_used += 45

    return plan


In [75]:

subjects = [
    {
        "subject": "Python",
        "topic": "Loops",
        "exam_date": "2026-10-15",
        "difficulty": 4
    },
    {
        "subject": "Mathematics",
        "topic": "Matrices",
        "exam_date": "2026-10-17",
        "difficulty": 3
    },
    {
        "subject": "DBMS",
        "topic": "SQL Basics",
        "exam_date": "2026-10-19",
        "difficulty": 5
    }
]

plan = generate_study_plan(subjects, daily_hours=2)

for task in plan:
    print(task)


{'date': '2026-10-09', 'subject': 'Python', 'topic': 'Loops', 'duration_minutes': 45, 'priority': 'High'}
{'date': '2026-10-09', 'subject': 'Mathematics', 'topic': 'Matrices', 'duration_minutes': 45, 'priority': 'Medium'}
{'date': '2026-10-10', 'subject': 'DBMS', 'topic': 'SQL Basics', 'duration_minutes': 45, 'priority': 'High'}


In [76]:

# Create a table for generated timetable sessions
cursor.execute("""
CREATE TABLE IF NOT EXISTS planned_sessions (
    session_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    study_date TEXT NOT NULL,
    subject TEXT NOT NULL,
    topic TEXT NOT NULL,
    duration_minutes INTEGER NOT NULL,
    priority TEXT NOT NULL,
    completed INTEGER DEFAULT 0,
    UNIQUE(student_id, study_date, subject, topic),
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

# Save the generated timetable without duplicating sessions
for task in plan:
    cursor.execute("""
    INSERT OR IGNORE INTO planned_sessions
    (student_id, study_date, subject, topic,
     duration_minutes, priority)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        student_id,
        task["date"],
        task["subject"],
        task["topic"],
        task["duration_minutes"],
        task["priority"]
    ))

conn.commit()

print("Timetable saved successfully!")


Timetable saved successfully!


In [77]:

cursor.execute("""
SELECT study_date, subject, topic,
       duration_minutes, priority, completed
FROM planned_sessions
WHERE student_id = ?
ORDER BY study_date
""", (student_id,))

saved_sessions = cursor.fetchall()

print("MY STUDY TIMETABLE")
print("-" * 65)

for session in saved_sessions:
    print(
        f"Date: {session[0]} | "
        f"Subject: {session[1]} | "
        f"Topic: {session[2]} | "
        f"Duration: {session[3]} minutes | "
        f"Priority: {session[4]} | "
        f"Completed: {session[5]}"
    )


MY STUDY TIMETABLE
-----------------------------------------------------------------
Date: 2026-10-09 | Subject: Mathematics | Topic: Matrices | Duration: 45 minutes | Priority: Medium | Completed: 0
Date: 2026-10-09 | Subject: Python | Topic: Loops | Duration: 45 minutes | Priority: High | Completed: 0
Date: 2026-10-10 | Subject: DBMS | Topic: SQL Basics | Duration: 45 minutes | Priority: High | Completed: 0


In [ ]:

cursor.execute("""
SELECT COUNT(*), COALESCE(SUM(completed), 0)
FROM planned_sessions
WHERE student_id = ?
""", (student_id,))

total, completed = cursor.fetchone()

percentage = completed / total * 100 if total else 0

print("Total sessions:", total)
print("Completed sessions:", completed)
print(f"Progress: {percentage:.1f}%")
